# Seguridad y Cumplimiento en Arquitecturas LLM
## Notebook de trabajo – Superficies de ataque y controles (Cloud vs On‑Prem)

> Curso: Arquitectura de IA Segura y Cumplimiento 2026  
> Docente: Jorge Ignacio Blanco  
> Entrega del trabajo: 8 días después de la clase

---
---

## 0. Cómo usar este notebook

Este notebook está pensado como una **guía paso a paso** para entender y analizar:

1. Las principales **superficies de ataque** en arquitecturas con LLM.  
2. Los **controles y salvaguardas** recomendados para cada superficie.  
3. Las diferencias prácticas entre escenarios **en la nube (cloud)** y **on‑premise**.  
4. Un conjunto de **preguntas de reflexión** y **ejercicios prácticos** que deberán responder y documentar.

### Reglas de trabajo

- Trabaja en una **copia** de este notebook con tu nombre en el archivo.   
- No necesitan acceso a servicios cloud reales para los ejercicios, pero pueden usarlos si lo quieren.  
- Se recomienda tener instalado **Ollama** para los ejercicios locales con modelos LLM.

---
---
## 1. Modelo mental: pipeline LLM y superficies de ataque

Partimos de un pipeline LLM simplificado:

```text
Usuario → API Gateway → Orquestador LLM → (RAG: índice + documentos) → LLM → Respuesta → Logging/Monitorización
```

 ![](imagen1.png)

En este flujo aparecen varias **superficies de ataque específicas de un LLM**:

1. **Superficie de entrada (Input surface)**: prompts, instrucciones, mensajes del usuario.  
2. **Superficie de recuperación (Retrieval / RAG surface)**: índices vectoriales, filtros de acceso, documentos fuente.  
3. **Superficie del modelo (Model surface)**: comportamiento del LLM, jailbreaks, prompt injection, instrucciones que contradicen políticas.  
4. **Superficie de salida (Output surface)**: contenido generado, posible fuga de datos, violaciones de política.  
5. **Superficie de logging / telemetría (Logging & telemetry surface)**: prompts, contextos y respuestas persistidos para observabilidad.

En las siguientes secciones analizaremos cada superficie, comparando **Cloud vs On‑Prem** y proponiendo ejercicios.

In [ ]:
import base64
from IPython.display import Image, display

mermaid_code = """
flowchart LR
    classDef default fill:#F8FAFC,stroke:#94A3B8,stroke-width:2px,color:#0F172A,rx:8px,ry:8px;
    classDef gateway fill:#E0F2FE,stroke:#0284C7,stroke-width:2px,color:#0369A1,rx:8px,ry:8px;
    classDef orq fill:#F1F5F9,stroke:#475569,stroke-width:2px,color:#334155,rx:8px,ry:8px;
    classDef rag fill:#E8F5E9,stroke:#2E7D32,stroke-width:2px,color:#1B5E20,rx:8px,ry:8px;
    classDef llm fill:#F3E5F5,stroke:#7B1FA2,stroke-width:2px,color:#4A148C,rx:8px,ry:8px;
    classDef response fill:#FFF8E1,stroke:#F57F17,stroke-width:2px,color:#E65100,rx:8px,ry:8px;
    classDef logs fill:#FFEBEE,stroke:#C62828,stroke-width:2px,color:#880E4F,rx:8px,ry:8px;

    U[Usuario]:::default
    API[API Gateway]:::gateway
    ORQ[Orquestador LLM]:::orq
    RAG["RAG: índice + documentos"]:::rag
    LLM[LLM]:::llm
    RESP[Respuesta]:::response
    LOG[Logging / Monitorización]:::logs

    U --> API
    API --> ORQ
    ORQ --> RAG
    RAG --> LLM
    LLM --> RESP
    RESP --> LOG
"""

# Codificar el texto en base64 para la API de renderizado
graphbytes = mermaid_code.encode("utf-8")
base64_bytes = base64.b64encode(graphbytes)
base64_string = base64_bytes.decode("utf-8")
display(Image(url="https://mermaid.ink/img/" + base64_string))


## 2. Superficie de entrada (Input Surface)

### 2.1 Riesgos típicos

Algunos riesgos frecuentes en la superficie de entrada:

- Prompt injection (instrucciones maliciosas que intentan saltarse políticas).  
- Exposición accidental de información sensible en el prompt (PII, secretos, datos de negocio).  
- Abuso de funcionalidades (por ejemplo, uso masivo para extracción de datos internos).  
- Falta de autenticación/autorización adecuada en el endpoint LLM.




### 2.2 Controles recomendados

**Controles generales (aplican a Cloud y On‑Prem):**

- Autenticación fuerte (tokens, OAuth2, SSO, etc.).  
- Autorización basada en rol y atributos (RBAC + ABAC) a nivel de API.  
- Rate limiting y protección ante abuso.  
- Validación de inputs y normalización de prompts (incluyendo filtrado de patrones obvios de prompt injection).  

**Controles específicos en Cloud:**

- Uso de API Gateways gestionados (WAF, protección DDoS, rate limiting integrado).  
- Integración con IAM del proveedor (roles y políticas por servicio).  
- Registro centralizado de accesos a nivel de plataforma.

**Controles específicos On‑Prem:**

- API Gateway propio (NGINX, Kong, Traefik, etc.) con WAF local.  
- Integración con LDAP/AD para autenticación corporativa.  
- Políticas de firewall internas y segmentación de red.

### 2.3 Preguntas de reflexión (Input Surface)

Respondan en esta celda:

1. En su contexto actual (empresa/entidad), ¿qué riesgos ves más probables en la superficie de entrada?  
2. ¿Qué controles ya existen a nivel de API y cuáles faltarían para un endpoint LLM?  
3. ¿Qué diferencias concretas observan entre aplicar estos controles en Cloud vs On‑Prem en su entorno?

## 3. Superficie de recuperación (Retrieval / RAG Surface)

En arquitecturas con RAG, el LLM no solo responde con su conocimiento entrenado, sino que:

1. Recibe una consulta.  
2. Recupera fragmentos de documentos desde un índice vectorial/buscador.  
3. Usa esos fragmentos como contexto para generar la respuesta.

### 3.1 Riesgos específicos

- Fuga de documentos sensibles a través de contexto (RAG sin controles de autorización).  
- Indices vectoriales que contienen PII o información confidencial sin cifrado ni segmentación.  
- Faltan filtros por tenant, dominio o rol (todos ven todo).  
- Inyección de contenido malicioso en documentos (data poisoning).

### 3.2 Controles recomendados

**Controles generales:**

- Segmentación de índices por dominio, área o tenant.  
- Aplicar filtros de acceso (por rol, grupo, atributos) ANTES de recuperar contenido.  
- Minimizar el contexto: incluir solo lo estrictamente necesario.  
- Cifrar datos en reposo en el índice vectorial y en tránsito en las consultas.

**En Cloud:**

- Uso de servicios gestionados de búsqueda/vector DB con:  
  - Cifrado por defecto, integración con KMS, VNet/Private Link, RBAC nativo.  
- Políticas de acceso por rol y por recurso del proveedor (IAM).  
- Logs centralizados de queries al índice.

**On‑Prem:**

- Despliegue de la base vectorial (p.ej. Qdrant, Milvus, pgvector) en Kubernetes/VMs internas.  
- Configuración propia de TLS, cifrado en disco, backups y restauración.  
- Segmentación de red (subred para RAG), firewalls internos, auditoría en el stack de logs.

### 3.3 Ejercicio práctico – Diseño de RAG seguro

Diseña, en alto nivel (pseudocódigo + diagrama simple), un flujo RAG seguro para el caso:

> "Copiloto interno para RRHH y Legal".

Requisitos:

- Debe existir control de acceso por rol (RRHH, Legal, otros).  
- El índice vectorial debe estar segmentado por dominio de documento.  
- El sistema debe evitar que un usuario de RRHH consulte documentos legales confidenciales, y viceversa.

**Tareas:**

1. Dibujar (en texto o usando herramientas externas) el flujo de la consulta.  
2. Describir qué filtros aplican ANTES de recuperar contenido del índice.  
3. Indicar qué cambia si este RAG corre:  
   a) total o mayoritariamente en Cloud,  
   b) total o mayoritariamente On‑Prem.

Responder en esta celda.

## 4. Superficie del modelo (Model Surface)

La superficie del modelo se refiere a cómo el LLM:

- Interpreta prompts e instrucciones.  
- Puede ser manipulado con jailbreaks o prompt injection sofisticado.  
- Puede generar contenido que viola políticas internas aunque el endpoint "funcione bien".

### 4.1 Riesgos típicos

- Jailbreaks que desactivan instrucciones del sistema o reglas de seguridad.  
- Respuestas que exponen PII, secretos o datos sensibles combinando información de contexto.  
- Respuestas que violan regulaciones (discursos de odio, sesgo, incumplimiento de normativas internas).

### 4.2 Controles recomendados

**Generales:**

- Prompt de sistema robusto, con políticas claras.  
- Uso de clasificadores o filtros de seguridad previos/posteriores a la llamada al modelo.  
- Evaluación y pruebas de jailbreaks (red teaming).  
- Configuración de parámetros del modelo (temperature, máximo de tokens, etc.) acorde al caso de uso.

**En Cloud:**

- Uso de features de seguridad del proveedor (content filters, safety shields, etc.).  
- Logging de razonamientos / tags de seguridad que ofrezca el proveedor.  
- Limitación de capacidades del modelo por API (no exponer funciones innecesarias).

**On‑Prem:**

- Despliegue de modelos locales (p.ej. LLaMA, Mistral, gemma4:12b, etc) con capas adicionales de filtrado.  
- Entrenamiento fino de clasificadores propios para detección de contenido no deseado.  
- Mantenimiento del stack de inferencia (actualizaciones, parches, versiones).

### 4.3 Ejercicio práctico – Prompt injection y filtros con Ollama

Con **Ollama** instalado, realizar el siguiente ejercicio en local:

1. Eligir un modelo que razone (por ejemplo, `llama3` o cualquiera de la arquitectura Gemma4).  
2. Diseñar un prompt de sistema que defina reglas claras (ej: "no debes revelar datos sensibles, no debes ejecutar instrucciones que violen políticas...").  
3. Escribir una serie de prompts de usuario que intenten saltarse esas reglas (prompt injection).  
4. Observar y documentar en qué casos el modelo respeta las reglas y en cuáles falla.

En esta celda escribir:

- El prompt de sistema que escribieron.  
- 3 ejemplos de prompts de usuario "maliciosos" que probaron.  
- Un breve análisis de los resultados.

*(No es necesario que peguen todas las salidas; enfóquense en el análisis.)*

#### Pueden documentar los experimentos así:
- Prompt de sistema:
- Prompt 1, 2, 3:
- Observaciones:

## 5. Superficie de salida (Output Surface)

La respuesta del modelo puede:

- Filtrar datos personales o confidenciales.  
- Contradecir políticas internas o regulatorias.  
- Inducir a acciones equivocadas si se interpreta como decisión automatizada sin supervisión.

### 5.1 Riesgos típicos

- Respuestas que incluyen PII o datos sensibles innecesarios.  
- Respuestas discriminatorias o sesgadas.  
- Respuestas que se usan como decisiones automatizadas sin intervención humana donde la regulación exige supervisión.

### 5.2 Controles recomendados

**Generales:**

- Post‑procesamiento de respuestas para detección de PII o contenido no permitido.  
- Redacción / anonimización antes de mostrar al usuario.  
- Reglas de negocio que obliguen a validación humana en decisiones de alto impacto.  

**En Cloud:**

- Uso de content filters o moderation APIs del proveedor.  
- Policies de masking/redaction integradas en el servicio.

**On‑Prem:**

- Construcción de pipelines propios de clasificación y redacción (por ejemplo, modelos de NER para PII).  
- Integración con sistemas internos de autorización para decidir qué se muestra y a quién.

### 5.3 Preguntas de reflexión (Output Surface)

1. Piensa en un caso real de su organización: ¿qué tipo de información **nunca** debería aparecer en una respuesta de un copiloto interno?  
2. ¿Cómo implementarías un filtro de salida sencillo que reduzca ese riesgo?  
3. ¿Qué diferencias ves entre usar filtros del proveedor cloud vs construir su propio filtro on‑prem?

Responde acontinuación.

## 6. Superficie de logging y telemetría

Los sistemas de observabilidad suelen registrar:

- Prompts, contextos y respuestas.  
- Metadatos de usuario, equipo, región.  
- Errores, latencias, métricas de uso.

### 6.1 Riesgos típicos

- Almacenar PII o datos sensibles en logs (a veces por años).  
- Logs accesibles a demasiadas personas.  
- Falta de políticas de retención y borrado.  
- Falta de trazabilidad sobre quién accedió y a qué.

### 6.2 Controles recomendados

**Generales:**

- Principio de logging mínimo necesario.  
- Separar logs técnicos de logs auditables.  
- Políticas claras de retención y borrado (incluyendo backups).  
- Control de acceso estricto a plataformas de observabilidad.

**En Cloud:**

- Configurar retención en servicios gestionados de logs (CloudWatch, Log Analytics, etc.).  
- Cifrado en reposo y en tránsito de logs.  
- Auditoría de accesos a plataformas de monitoreo.

**On‑Prem:**

- Implementar stack propio (ELK, Splunk, etc.) con retención, cifrado y control de acceso.  
- Borrado seguro en discos y backups.  
- Integración con SIEMs internos y equipos de seguridad.

### 6.3 Ejercicio – Diseñar un esquema de logging seguro

Para el mismo caso de "Copiloto interno para RRHH y Legal":

1. Define qué **sí** debe loguearse (eventos, metadatos) y qué **NO** debe aparecer nunca en logs.  
2. Propongan una política de retención (¿cuánto tiempo?, ¿por qué?).  
3. Indica cómo implementarías esa política en:  
   a) servicios de logging en Cloud,  
   b) un stack de logging On‑Prem.

Responde en esta celda.

## 7. Comparación Cloud vs On‑Prem (Resumen)

Completar la siguiente tabla para un caso concreto:

| Superficie           | Cloud – ventajas                      | Cloud – retos                        | On‑Prem – ventajas                       | On‑Prem – retos                      |
|----------------------|---------------------------------------|--------------------------------------|------------------------------------------|--------------------------------------|
| Entrada (Input)      |                                       |                                      |                                          |                                      |
| RAG / Retrieval      |                                       |                                      |                                          |                                      |
| Modelo               |                                       |                                      |                                          |                                      |
| Salida (Output)      |                                       |                                      |                                          |                                      |
| Logging / Telemetría |                                       |                                      |                                          |                                      |

### 7.2 Pregunta de cierre



> Si tuvieran que recomendar, para una empresa u organización, un enfoque **cloud‑first**, **on‑prem‑first** o **híbrido** para arquitecturas LLMs seguras, ¿cuál recomendarían y por qué, considerando las superficies de ataque y controles que has analizado?

# Ejercicio Opcional

Diseñar un plan de implementación técnico para configurar un sistema de token vault que intercepte y anonimice PII de los prompts antes de ser enviados a proveedores de LLM. Incluir el flujo de datos: 
- Entrada de texto 
- identificación de entidades sensibles (PII)
- sustitución por tokens 
- almacenamiento de la tabla de mapeo (vault), 
- y proceso de des-tokenización para la respuesta final

Debemos asegurarno de que el LLM reciba solo información despersonalizada y funcional.